# Bronze - Releases Data

## Setup Environment and Prepare data 

In [ ]:
from letterboxd_data_pipeline.load_data import load_files_from_layer
from letterboxd_data_pipeline.explore_data.explore_data import is_possible_na

# bronze_df = load_files_from_layer(layer="bronze", file_list=["releases"])
bronze_releases_df = load_files_from_layer(layer="bronze", file_list=["releases.parquet"])["releases"]

Start loading data...
Loading: data/bronze/releases.parquet
Done!
Finish loading.


## Simple Explore

In [3]:
bronze_releases_df.describe(include="all")

,id,country,date,type,rating
count,1332782,1332782,1332782,1332782,333980
unique,826018,246,43108,6,284
top,1001112,USA,2006-01-01,Theatrical,NR
freq,251,320901,3050,750043,42499


In [4]:
bronze_releases_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1332782 entries, 0 to 1332781
Data columns (total 5 columns):
 #   Column   Non-Null Count    Dtype 
---  ------   --------------    ----- 
 0   id       1332782 non-null  object
 1   country  1332782 non-null  object
 2   date     1332782 non-null  object
 3   type     1332782 non-null  object
 4   rating   333980 non-null   object
dtypes: object(5)
memory usage: 50.8+ MB


In [5]:
bronze_releases_df.head(10)

,id,country,date,type,rating
0,1000001,Andorra,2023-07-21,Theatrical,None
1,1000001,Argentina,2023-07-20,Theatrical,ATP
2,1000001,Australia,2023-07-19,Theatrical,PG
3,1000001,Australia,2023-10-01,Digital,PG
4,1000001,Austria,2023-07-20,Theatrical,None
5,1000001,Austria,2023-07-21,Theatrical,None
6,1000001,Bahrain,2023-08-10,Theatrical,None
7,1000001,Belgium,2023-07-19,Theatrical,None
8,1000001,Bolivarian Republic of Venezuela,2023-07-20,Theatrical,None
9,1000001,Bolivia,2023-07-20,Theatrical,None


## Number of NA

In [6]:
bronze_releases_df.isna().sum()

id              0
country         0
date            0
type            0
rating     998802
dtype: int64

## Possible NA

In [7]:
possible_na_df = bronze_releases_df[
  is_possible_na(bronze_releases_df["country"]) | 
  is_possible_na(bronze_releases_df["date"]) |
  is_possible_na(bronze_releases_df["type"]) |
  is_possible_na(bronze_releases_df["rating"])
]
possible_na_df

,id,country,date,type,rating


## Movies with Most Country Releases

In [8]:
most_globe_releases_df = bronze_releases_df.copy(True)

most_globe_releases_df["country_count"] = most_globe_releases_df.groupby(by=["id"])[
    "country"
].transform(lambda x: x.nunique())


(most_globe_releases_df[["id", "country_count", "country"]]
.drop_duplicates()
.sort_values(["country_count", "id", "country"], ascending=[False, True, True]))

,id,country_count,country
33276,1000577,234,Afghanistan
33277,1000577,234,Albania
33278,1000577,234,Algeria
33279,1000577,234,American Samoa
33280,1000577,234,Andorra
...,...,...,...
1332777,1940967,1,USA
1332778,1940968,1,Sweden
1332779,1940969,1,France
1332780,1940970,1,France


## Movies with Type of Release per Country

In [9]:
release_types_df = bronze_releases_df.copy(True)

release_types_df.groupby(by=["id", "country"]).agg(
    unique_type_count=("type", "nunique")
).sort_values(by=["id","unique_type_count"], ascending=[True, False])

unique_type_count
id      country                       
1000001 USA                          6
        Puerto Rico                  5
        Germany                      3
        Japan                        3
        UK                           3
...                                ...
1940967 USA                          1
1940968 Sweden                       1
1940969 France                       1
1940970 France                       1
1940971 France                       1

[1214216 rows x 1 columns]